# Stream NISAR GCOV Data with `asf_search`, `s3fs`, and `xarray`

<br>

[`asf_search`](https://github.com/asfadmin/Discovery-asf_search) is an open source Python package for searching SAR data archived at {abbr}`ASF (Alaska Satellite Facility)`. This notebook demonstrates how to search NISAR GCOV with `asf_search`, stream them directly from S3 storage with `s3fs`, and load them into `xarray` data structures.

NISAR data products can be very large. It may be helpful to access the data directly from their S3 storage. This allows you to lazily load data into `xarray` data structures, subset, and perform operations on them with `xarray`, and then only the data you need into memory. This avoids downloading many large products to a storage volume, only to subset and delete most of them. The caveat is that you must have enough RAM to hold your final subset and also run your workflow.

:::{warning} You can only access NISAR data directly from an S3 bucket if you are working in the `us-west-2` AWS region

Often, this means that you are working on an AWS EC2 instance in the `us-west-2` region, though you could possibly be working with another in-region AWS service, such as Lambda, Fargate, SageMaker, etc...

You could also be working on an in-region, AWS-hosted JupyterHub service such as ASF OpenScienceLab.

**Direct S3 access is not possible from a Python script or Jupyter Notebook running directly on your computer or HPC.**

:::

<hr>

## Overview

1. [Prerequisites](asf_search_stream-prereqs)
1. [Search for data](asf_search_stream-search)
1. [(Option 1) Load a single GCOV product using a Python `with` statement](asf_search_stream-with)
1. [(Option 2) Load multiple GCOV Products at once](asf_search_stream-open_many)
1. [Stream a time series](asf_search_stream-ts)
1. [Close open files](asf_search_stream-close)
1. [Summary](asf_search_stream-summary)
1. [Resources and references](asf_search_stream-resources)

<hr>

(asf_search_stream-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 10 minutes

<hr>

(asf_search_stream-search)=
## 3. Search for data

Use `asf_search` to find GCOV data.

### 3a. Perform an `asf_search.search()` to identify your desired product URLs


In [ ]:
import os
import asf_search as asf
from datetime import datetime
from getpass import getpass

session = asf.ASFSession()

start_date = datetime(2025, 11, 22)
end_date = datetime(2025, 12, 5)
area_of_interest = "POLYGON((40.9131 12.3904,41.8891 12.3904,41.8891 13.2454,40.9131 13.2454,40.9131 12.3904))" # POINT or POLYGON as WKT (well-known-text)
pattern = r'^(?!.*QA_STATS).*'

opts=asf.ASFSearchOptions(**{
    "maxResults": 250,
    "intersectsWith": area_of_interest,
    "start": start_date,
    "end": end_date,
    "processingLevel": [
        "GCOV"
    ],
    "dataset": [
        "NISAR"
    ],
    "productionConfiguration": [
        "PR"
    ],
    'session': session,
})

response = asf.search(opts=opts)
hdf5_urls = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
hdf5_urls

### 3b. Use an Earthdata Login Bearer Token to get S3 bucket access credentials

To create or view a previously created Earthdata Login Bearer Token, sign into your [User Profile Page](https://urs.earthdata.nasa.gov/profile), click the "Generate Token" tab, and follow the provided instructions.

:::{note} Temporary S3 credentials

The S3 credentials acquired below are temporary and expire after 1 hour.
:::

In [ ]:
from getpass import getpass
import json
import s3fs
import urllib

token = getpass("Enter your EDL Bearer Token")
bucket = "sds-n-cumulus-prod-nisar-products"
prefix = "NISAR_L2_GCOV_BETA_V1"

event = {
    "CredentialsEndpoint": "https://nisar.asf.earthdatacloud.nasa.gov/s3credentials",
    "BearerToken": token,
    "Bucket": bucket,
    "Prefix": prefix,
    # "StaticPrefix": f"{prefix}_STATIC"
}

# Get temporary download credentials
tea_url = event["CredentialsEndpoint"]
bearer_token = event["BearerToken"]
req = urllib.request.Request(
    url=tea_url,
    headers={"Authorization": f"Bearer {bearer_token}"}
)
with urllib.request.urlopen(req) as f:
    creds = json.loads(f.read().decode())

fs = s3fs.S3FileSystem(
    key=creds["accessKeyId"],
    secret=creds["secretAccessKey"],
    token=creds["sessionToken"],
)

### 3c. Convert the hdf5 URLs into S3 urls

In [ ]:
from urllib.parse import urlparse

s3_urls = [f"s3://{bucket}/{'/'.join(urlparse(url).path.split('/')[2:])}" for url in hdf5_urls]
s3_urls

<hr>

(asf_search_stream-with)=
## 4. (Option 1) Load a single GCOV product using a Python `with` statement

 <br>

:::{note} This requires that you do all of your work on the dataset within the `with` block

Using `with` automatically closes an open file handle once out of scope of the `with` block. This is memory-safe but can also cause problems.

`with` statements are most appropriate when performing limited operations on a single data product. 

They are less useful when working with a time series, when you probably want to have multiple files open at once.
:::

### 4a. Open a GCOV file in a `with` block and compute the mean of an HHHH spatial subset

In [ ]:
%%time
import xarray as xr
import rioxarray

s3_url = s3_urls[0]

with fs.open(s3_url, "rb") as f:
    dt = xr.open_datatree(f, engine="h5netcdf", decode_timedelta=False, phony_dims="access")

    ### Perform any calculations and save any computed results for future access here, inside the `with` block ###
    frequencyA = dt["/science/LSAR/GCOV/grids/frequencyA"] # access the frequency A data
    projection = frequencyA.projection.attrs['epsg_code'].item() # access the GCOV product's projection
    hhhh = frequencyA.HHHH # access frequency A's HHHH band
    hhhh = hhhh.rio.write_crs(projection) # write the project to the HHHH data for easy lat/lon subsetting

    # subset the data
    subset_hhhh = hhhh.rio.clip_box(
        minx=40.8463, miny=13.2553,
        maxx=40.8574, maxy=13.2684, 
        crs="EPSG:4326"
    )

    # save the mean of HHHH subset for use outside of the `with` block
    subset_hhhh_mean = subset_hhhh.mean().to_numpy().item()
    print(f'subset_hhhh_mean: {subset_hhhh_mean}\n')

### 4b. View the `DataTree`

You can look at the `DataTree` outside the `with` block, but it only contains pointers to the data and the file is closed, so you can no longer access any of the data values to which it points

In [ ]:
dt

### 4c. Try (and fail) to compute a value in the `DataTree`

The file was closed upon exiting the `with` block so trying to compute a value from the `DataTree` will fail with a `ValueError: I/O operation on closed file.`

In [ ]:
dt.compute()

#### However, `subset_hhhh_mean` was computed and saved to memory inside the `with` block, so we can still see its value

In [ ]:
subset_hhhh_mean

<hr>

(asf_search_stream-open_many)=
## 5. (Option 2) Load multiple GCOV Products at once

:::{note} Lets speed things up a bit

You probably noticed that it took a long time to load the entire HDF5 file as a `DataTree` in the example above. You can speed this up by only accessing the HDF5 group that you need.

For the remaining examples in this notebook, we will access only the Frequency A data, stored in the group `/science/LSAR/GCOV/grids/frequencyA`.
:::

### 5a. Iterate through a list of HDF5 S3 bucket URLs, and open the `/science/LSAR/GCOV/grids/frequencyA` for each

This leaves the files open for later use, which means you must manually close them when finished in order to prevent memory leaks.

In [ ]:
%%time
import xarray as xr
import rioxarray

# Explore the DataTree rendering above in Step 4 for a complete list of available groups 
group_path = "/science/LSAR/GCOV/grids/frequencyA" # change this to any GCOV HDF5 group you wish

files = [fs.open(url, "rb") for url in s3_urls]

datatrees = [
    xr.open_datatree(
        f,
        engine="h5netcdf",
        decode_timedelta=False,
        phony_dims="access",
        chunks="auto",
        group=group_path,
    )
    for f in files
]

### 5b. Since the files remain open, we can access still access their

Iterate through the `DataTrees`, calculating subset HHHH mean values

In [ ]:
for tree in datatrees:
    projection = tree.projection.attrs['epsg_code'].item()
    hhhh = tree.HHHH
    hhhh = hhhh.rio.write_crs(projection)
    
    subset_hhhh = hhhh.rio.clip_box(
        minx=40.8463, miny=13.2553,
        maxx=40.8574, maxy=13.2684, 
        crs="EPSG:4326"
    )
    subset_hhhh_mean = subset_hhhh.mean().to_numpy().item()
    print(subset_hhhh_mean)

<hr>

(asf_search_stream-ts)=
## 6. Stream a time series

Use the multiple files we opened in Step 5 to create a GCOV time series

### 6a. Define a function to extract datetimes for a `time` dimension

In [ ]:
import re
from urllib.parse import urlparse
from datetime import datetime
from pathlib import PurePosixPath

NISAR_TS_RE = re.compile(r"_(\d{8}T\d{6})_")

def nisar_start_time_from_url(s3_url: str) -> datetime:
    path = urlparse(s3_url).path
    name = PurePosixPath(path).name
    
    m = NISAR_TS_RE.search(name)
    if not m:
        raise ValueError(f"No NISAR timestamp found in: {s3_url}")
    
    return datetime.strptime(m.group(1), "%Y%m%dT%H%M%S")

### 6b. Create a list of datetimes for a `time` dimension

In [ ]:
dts = [nisar_start_time_from_url(url) for url in s3_urls]
dts

### 6c. Create an `xarray.DataArray` containing a `time` dimension for each open GCOV group

In [ ]:
dataarrays = [
    tree.ds.assign_coords(time=dt).expand_dims(time=1)
    for dt, tree in zip(dts, datatrees)
]

dataarrays[0].time

### 6d. Concatenate the `xarray.Dataset`s into a single `xarray.Dataset` time series 

In [ ]:
ts = xr.concat(dataarrays, dim="time")
ts

<hr>

(asf_search_stream-close)=
## 7. Close open files

Close all open files to prevent memory leaks.

### 7a. Close the files

In [ ]:
for f in files:
    f.close()

#### 7b. With closed files, we can no longer compute values in the `xarray` data structures that we have created

The cell below will return a `ValueError: I/O operation on closed file.`

In [ ]:
ts.compute()

<hr>

(asf_search_stream-summary)=
## 8. Summary
You now have the tools and knowledge that you need to search with `asf_search`, generate temporary S3 bucket credentials from an Earthdata Login Bearer Token, stream data from S3 with `s3fs`, and load them into `xarray` data structures.

<hr>

(asf_search_stream-resources)=
## 9. Resources and references
 - [asf_search](https://github.com/asfadmin/Discovery-asf_search)
 - [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/aws-s3-access/)
 
**Author:** Alex Lewandowski